# 02 — Pré-processamento de Texto

O **pré-processamento** é a etapa que transforma texto bruto em uma forma limpa e normalizada, pronta para ser vetorizada. Textos de redes sociais contêm muito ruído — pontuações, números, palavras irrelevantes — que devem ser removidos para que os modelos aprendam padrões realmente significativos.

Neste notebook:
1. Visualizamos cada etapa do pipeline individualmente com um texto de exemplo
2. Aplicamos o pipeline completo ao dataset
3. Realizamos a divisão Treino/Teste
4. Salvamos os dados processados para uso na vetorização

---
**Colunas:** `text` (texto) | `label` (0 = sem ideação · 1 = com ideação)

## 1. Importações

In [1]:
%matplotlib inline

import sys
sys.path.append('..')

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    limpar_texto, tokenizar, remover_stopwords,
    aplicar_stemming, preprocessar, preprocessar_coluna
)

COLUNA_TEXTO = 'text'
COLUNA_LABEL = 'label'

---
## 2. Visualização do Pipeline — Etapa por Etapa

Antes de aplicar o pré-processamento ao dataset inteiro, vamos entender o que acontece em cada etapa usando um texto de exemplo.

O pipeline segue a seguinte sequência:

```
Texto bruto
    │
    ▼
① Limpeza        →  minúsculas, remove URLs, pontuação, números
    │
    ▼
② Tokenização    →  divide em lista de palavras (tokens)
    │
    ▼
③ Stopwords      →  remove palavras sem valor semântico
                    (negações como "não", "nunca" são preservadas)
    │
    ▼
④ Stemming       →  reduz palavras à raiz morfológica (RSLP)
    │
    ▼
Texto processado  →  pronto para vetorização
```

In [2]:
# Texto de exemplo — troque por qualquer outro para testar!
texto_exemplo = "Estou sofrendo, não aguento mais! Já faz 2 meses que penso em acabar com tudo..."

print("=" * 60)
print("TEXTO ORIGINAL")
print("=" * 60)
print(texto_exemplo)

TEXTO ORIGINAL
Estou sofrendo, não aguento mais! Já faz 2 meses que penso em acabar com tudo...


### Etapa 1 — Limpeza do Texto

Nesta etapa aplicamos:
- **Minúsculas**: padroniza o texto ("Sofrendo" e "sofrendo" viram o mesmo token)
- **Remoção de URLs**: links não carregam significado semântico útil
- **Remoção de menções/hashtags**: `@usuario` e `#tag` são ruídos em dados de redes sociais
- **Remoção de pontuação**: vírgulas, exclamações etc. não contribuem para a classificação
- **Remoção de números**: dígitos isolados raramente são informativos

In [3]:
etapa1 = limpar_texto(texto_exemplo)

print("ETAPA 1 — Limpeza")
print("-" * 60)
print(f"Entrada : {texto_exemplo}")
print(f"Saída   : {etapa1}")

ETAPA 1 — Limpeza
------------------------------------------------------------
Entrada : Estou sofrendo, não aguento mais! Já faz 2 meses que penso em acabar com tudo...
Saída   : estou sofrendo não aguento mais já faz meses que penso em acabar com tudo


### Etapa 2 — Tokenização

**Tokenização** divide o texto em unidades menores chamadas **tokens** (geralmente palavras). Utilizamos o tokenizador do NLTK configurado para o **português**, que lida corretamente com contrações e particularidades da língua.

In [4]:
etapa2 = tokenizar(etapa1)

print("ETAPA 2 — Tokenização")
print("-" * 60)
print(f"Entrada : {etapa1}")
print(f"Saída   : {etapa2}")
print(f"Total de tokens: {len(etapa2)}")

ETAPA 2 — Tokenização
------------------------------------------------------------
Entrada : estou sofrendo não aguento mais já faz meses que penso em acabar com tudo
Saída   : ['estou', 'sofrendo', 'não', 'aguento', 'mais', 'já', 'faz', 'meses', 'que', 'penso', 'em', 'acabar', 'com', 'tudo']
Total de tokens: 14


### Etapa 3 — Remoção de Stopwords

**Stopwords** são palavras muito frequentes que não carregam significado semântico relevante ("de", "que", "eu", "já", "mais", etc.).

> ⚠️ **Decisão de projeto importante:** termos de negação como **"não", "nunca" e "jamais"** são preservados intencionalmente. No contexto de saúde mental, essas palavras invertem a polaridade da frase — "não quero morrer" é semanticamente oposto a "quero morrer". A remoção indevida dessas palavras degrada severamente o desempenho dos classificadores.

In [5]:
etapa3 = remover_stopwords(etapa2)

removidas = set(etapa2) - set(etapa3)

print("ETAPA 3 — Remoção de Stopwords")
print("-" * 60)
print(f"Entrada       : {etapa2}")
print(f"Saída         : {etapa3}")
print(f"Tokens removidos ({len(removidas)}): {removidas}")
print(f"Tokens restantes: {len(etapa3)} (de {len(etapa2)} originais)")

ETAPA 3 — Remoção de Stopwords
------------------------------------------------------------
Entrada       : ['estou', 'sofrendo', 'não', 'aguento', 'mais', 'já', 'faz', 'meses', 'que', 'penso', 'em', 'acabar', 'com', 'tudo']
Saída         : ['sofrendo', 'não', 'aguento', 'faz', 'meses', 'penso', 'acabar', 'tudo']
Tokens removidos (6): {'já', 'com', 'estou', 'que', 'em', 'mais'}
Tokens restantes: 8 (de 14 originais)


### Etapa 4 — Stemming (RSLP)

**Stemming** reduz cada palavra à sua **raiz morfológica**, agrupando variações:
- "sofrendo", "sofreu", "sofrimento" → `sofr`
- "pensando", "pensou", "pensamento" → `pens`

Usamos o **RSLP** (*Removedor de Sufixos da Língua Portuguesa*), desenvolvido especificamente para o português.

> As raízes geradas podem não ser palavras reais — isso é esperado. O que importa é que variações de uma mesma palavra sejam agrupadas sob o mesmo token.

In [6]:
etapa4 = aplicar_stemming(etapa3)

print("ETAPA 4 — Stemming (RSLP)")
print("-" * 60)
print(f"Entrada : {etapa3}")
print(f"Saída   : {etapa4}")
print("\nMapeamento token → raiz:")
for original, raiz in zip(etapa3, etapa4):
    marcador = "✓" if original == raiz else "→"
    print(f"  {original:20s} {marcador}  {raiz}")

ETAPA 4 — Stemming (RSLP)
------------------------------------------------------------
Entrada : ['sofrendo', 'não', 'aguento', 'faz', 'meses', 'penso', 'acabar', 'tudo']
Saída   : ['sofr', 'não', 'aguent', 'faz', 'mes', 'pens', 'acab', 'tud']

Mapeamento token → raiz:
  sofrendo             →  sofr
  não                  ✓  não
  aguento              →  aguent
  faz                  ✓  faz
  meses                →  mes
  penso                →  pens
  acabar               →  acab
  tudo                 →  tud


### Resumo do Pipeline Completo

In [7]:
resultado_final = preprocessar(texto_exemplo, usar_stemming=True)

print("RESUMO DO PIPELINE")
print("=" * 60)
print(f"① Texto original  : {texto_exemplo}")
print(f"② Após limpeza    : {etapa1}")
print(f"③ Após tokenização: {etapa2}")
print(f"④ Após stopwords  : {etapa3}")
print(f"⑤ Após stemming   : {etapa4}")
print("-" * 60)
print(f"✅ Texto final     : {resultado_final}")

RESUMO DO PIPELINE
① Texto original  : Estou sofrendo, não aguento mais! Já faz 2 meses que penso em acabar com tudo...
② Após limpeza    : estou sofrendo não aguento mais já faz meses que penso em acabar com tudo
③ Após tokenização: ['estou', 'sofrendo', 'não', 'aguento', 'mais', 'já', 'faz', 'meses', 'que', 'penso', 'em', 'acabar', 'com', 'tudo']
④ Após stopwords  : ['sofrendo', 'não', 'aguento', 'faz', 'meses', 'penso', 'acabar', 'tudo']
⑤ Após stemming   : ['sofr', 'não', 'aguent', 'faz', 'mes', 'pens', 'acab', 'tud']
------------------------------------------------------------
✅ Texto final     : sofr não aguent faz mes pens acab tud


---
## 3. Aplicação ao Dataset Completo

Agora que entendemos cada etapa, aplicamos o pipeline a todas as amostras do dataset.

> ⏳ Esta célula pode demorar alguns minutos dependendo do hardware.

In [8]:
df = pd.read_csv('../data/raw/Boamente_Atualizado_Janeiro_2026.csv')
print(f'Dataset carregado: {df.shape}')

print('Aplicando pré-processamento...')
df['texto_processado'] = preprocessar_coluna(df[COLUNA_TEXTO], usar_stemming=True)
print('Concluído!')

df[['text', 'texto_processado']].head(10)

Dataset carregado: (3777, 2)
Aplicando pré-processamento...
Concluído!


,text,texto_processado
0,Aquela vontade de acabar com a minha vida voltou,vontad acab vid volt
1,to triste e com vontade de acabar com a minha ...,trist vontad acab vid
2,Corinthians ta querendo acabar com minha vida ...,corinth quer acab vid
3,Alguém poderia por favor me dar um tiro a acab...,alguém pod favor dar tir acab vid
4,TAYLOR TU VAI acabar com a minha vida MULHER,tayl vai acab vid mulh
5,Eu vou assistir o jogo todo puto mais uma vez....,vou assist jog tod put vez velh maldit vai con...
6,Alguem pode me.indicar uma série uma boa q vai...,algu pod meindic séri boa vai acab vid soc pró...
7,tenho 76 slides p estudar obg faculdade por ac...,slid estud obg faculdad acab vid soc
8,eu nao concordei com um tweet da fernanda agor...,nao concord tweet fernand agor desenterr hit p...
9,sessão de foto pra acabar com a minha vida ess...,sess fot pra acab vid fud


## 4. Divisão Treino/Teste

Dividimos o dataset em **80% treino** e **20% teste**. O parâmetro `stratify=y` garante que a proporção das classes seja mantida em ambos os conjuntos, evitando que um subconjunto fique desbalanceado por acaso.

O `random_state=42` fixa a aleatoriedade, garantindo **reprodutibilidade** — qualquer pessoa que execute este notebook obterá exatamente os mesmos subconjuntos.

In [9]:
X = df['texto_processado']
y = df[COLUNA_LABEL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino : {len(X_train)} amostras')
print(f'Teste  : {len(X_test)} amostras')
print(f'\nDistribuição treino:\n{y_train.value_counts()}')
print(f'\nDistribuição teste:\n{y_test.value_counts()}')

Treino : 3021 amostras
Teste  : 756 amostras

Distribuição treino:
label
0    2149
1     872
Name: count, dtype: int64

Distribuição teste:
label
0    538
1    218
Name: count, dtype: int64


## 5. Salvar os Dados Processados

Salvamos os textos pré-processados (ainda como strings, sem vetorizar) para serem carregados pelo próximo notebook.

A vetorização (BoW, TF-IDF, Word2Vec, GloVe) é tratada separadamente em **`03_vectorization.ipynb`**.

In [10]:
joblib.dump((X_train, X_test, y_train, y_test), '../data/processed/splits_texto.joblib')

['../data/processed/splits_texto.joblib']